In [30]:
import pandas as pd

telemetry_path = 'data/farm/tractor_telemetry.csv.gz' 
df = pd.read_csv(telemetry_path)
df['ts'] = pd.to_datetime(df['ts'])
df.head()

,ts,tractor_id,lat,lon,speed_mps,cross_track_error_m,num_sats,mean_cn0_dbhz,min_cn0_dbhz,fix_state,base_link_ok
0,2026-05-04 00:00:00+00:00,T1,46.900041,-97.203944,4.5,0.009,12,45.1,40.4,RTK_FIXED,True
1,2026-05-04 00:00:01+00:00,T1,46.900081,-97.203944,4.5,0.012,11,45.7,39.7,RTK_FIXED,True
2,2026-05-04 00:00:02+00:00,T1,46.900121,-97.203944,4.5,0.004,13,45.8,40.3,RTK_FIXED,True
3,2026-05-04 00:00:03+00:00,T1,46.900162,-97.203944,4.5,0.008,12,45.0,40.7,RTK_FIXED,True
4,2026-05-04 00:00:04+00:00,T1,46.900202,-97.203944,4.5,0.019,13,45.8,41.1,RTK_FIXED,True


In [31]:
# High drift while the trractor still thinks it has a perfect RTK fiz
spoofing_sus = df[(df['cross_track_error_m'] > 0.5) & 
                  (df['fix_state'] == 'RTK_FIXED') & 
                  (df['mean_cn0_dbhz'] > 40)]
spoofing_suspects.head()

,ts,tractor_id,lat,lon,speed_mps,cross_track_error_m,num_sats,mean_cn0_dbhz,min_cn0_dbhz,fix_state,base_link_ok
77332,2026-05-04 21:28:52+00:00,T1,46.907185,-97.198896,4.5,0.538,13,42.2,26.4,RTK_FIXED,True
88306,2026-05-05 00:31:46+00:00,T1,46.905233,-97.198574,4.5,0.519,11,45.0,39.0,RTK_FIXED,True
88309,2026-05-05 00:31:49+00:00,T1,46.905354,-97.198574,4.5,0.538,12,44.7,40.6,RTK_FIXED,True
88316,2026-05-05 00:31:56+00:00,T1,46.905637,-97.198574,4.5,0.503,12,45.6,40.1,RTK_FIXED,True
88317,2026-05-05 00:31:57+00:00,T1,46.905678,-97.198574,4.5,0.528,13,44.9,39.5,RTK_FIXED,True


In [32]:
# Check for a dropped RTK base station connection
link_faults = df[df['base_link_ok'] == False]

# Or, check for signal drops specific to just Tractor 1
t1_faults = df[(df['tractor_id'] == 'T1') & (df['mean_cn0_dbhz'] < 30)]

In [33]:
# Filter for severe signal degradation across the board
signal_loss = df[(df['mean_cn0_dbhz'] < 35) | (df['fix_state'] != 'RTK_FIXED')]

In [34]:
# Find brief spikes in error
high_error = df[df['cross_track_error_m'] > 0.3]

# Round the coordinates to group them and find recurring trouble spots
high_error_locations = high_error.round({'lat': 4, 'lon': 4})
high_error_locations['lat_lon'] = high_error_locations['lat'].astype(str) + ", " + high_error_locations['lon'].astype(str)

# See which physical locations have the most errors
high_error_locations['lat_lon'].value_counts().head()

lat_lon
46.9071, -97.1998    38
46.9071, -97.2019    35
46.9031, -97.2019    34
46.9071, -97.2013    33
46.9071, -97.1961    33
Name: count, dtype: int64